# Session 3: Visualization, regression tables, coefficient plots

**TODAY:** Bar plots, Scatter plots, *[Break]*, Tables, Coefficient plots.

Python's plotting is built in layers. **matplotlib** is the foundation: you create a *figure* containing one or more *axes* (the plotting areas) and draw on them. **seaborn** sits on top of matplotlib and produces common statistical plots in one line, returning the axes so you can keep customizing them. (R users: the closest thing to `ggplot2`'s layered grammar is the package `plotnine`, which copies its syntax almost exactly. Here we use the standard matplotlib + seaborn pair, which is what most Python data code uses.)

### Bar plots

In [ ]:
from pathlib import Path
import calendar

import pandas as pd

MONTHS = list(calendar.month_name)[1:]   # ["January", "February", ..., "December"]


def find_data_file(name):
    """Look for data/<name> in this folder, in the folders above it, and one level below."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / "data" / name
        if candidate.exists():
            return candidate
    for candidate in sorted(here.glob(f"*/data/{name}")):
        return candidate
    raise FileNotFoundError(
        f"Could not find data/{name}. Python is currently looking from: {here}\n"
        "Fix: put the 'data' folder next to this notebook (or in a folder above it)."
    )

def load_seatbelts():
    """The monthly data frame we build in Session 1 -- rebuilt on the spot if the file is missing."""
    try:
        df = pd.read_csv(find_data_file("seatbelts_monthly.csv"), parse_dates=["date"])
    except FileNotFoundError:
        try:
            df = pd.read_csv(find_data_file("Seatbelts.csv"))
        except FileNotFoundError:
            import statsmodels.api as sm   # last resort: download the same data (needs internet)
            try:
                df = sm.datasets.get_rdataset("Seatbelts", "datasets").data
            except Exception as err:
                raise FileNotFoundError(
                    "Could not find data/Seatbelts.csv and could not download it either. "
                    "Keep the 'data' folder next to the notebooks (or connect to the internet)."
                ) from err
        df["date"] = pd.date_range("1969-01-01", periods=len(df), freq="MS")
        df["month"] = df["date"].dt.month_name()
    df["month"] = pd.Categorical(df["month"], categories=MONTHS, ordered=True)
    return df


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")   # a clean look and a colour-blind-friendly palette
plt.rcParams["figure.figsize"] = (8, 4.5)

seatbelts_monthly = load_seatbelts()
seatbelts_monthly.info()

In [ ]:
# Create a basic bar plot -- this is the same seasonality pattern we found with groupby().mean()
# in Session 2, now as a picture
fig, ax = plt.subplots()
sns.barplot(data=seatbelts_monthly, x="month", y="DriversKilled", errorbar=None, color="steelblue", ax=ax)
ax.set(title="Basic Bar Plot", xlabel="Month", ylabel="Mean drivers killed per month")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
# estimator: sns.barplot draws the MEAN of y for each x by default (the `estimator` argument). If you want the
# bars to show totals instead, use estimator="sum" -- but think about which question you are asking:
# "how many deaths happened in total?" or "how many deaths happen in a typical month?". The next plot
# shows why that matters.

In [ ]:
# Create a grouped bar plot
# LawPeriod is a derived category (before/after the seatbelt law) -- it gives us a second
# grouping variable to colour by
seatbelts_monthly["LawPeriod"] = np.where(seatbelts_monthly["law"] == 1, "Post-law", "Pre-law")

fig, ax = plt.subplots()
sns.barplot(data=seatbelts_monthly, x="month", y="DriversKilled", hue="LawPeriod", errorbar=None, ax=ax)
# hue= puts the groups side by side within each month (R/ggplot: position = "dodge")
ax.set(title="Grouped Bar Plot", xlabel="Month", ylabel="Mean drivers killed per month")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
# NB: we plot MEANS. There are 14 pre-law months but only 2 post-law months for each calendar month, so if the
# bars were totals (sums), the pre-law bars would be about 9 times taller than the post-law ones simply
# because there are more of them -- a misleading picture. Means make the two groups comparable.

In [ ]:
# Eliminate NAs
# seatbelts_monthly itself has no missing data, so for teaching purposes we'll pretend
# a few months' LawPeriod label didn't get recorded
rng = np.random.default_rng(1)
missing_rows = rng.choice(len(seatbelts_monthly), size=4, replace=False)
seatbelts_monthly.loc[seatbelts_monthly.index[missing_rows], "LawPeriod"] = np.nan

clean_seatbelts = seatbelts_monthly.dropna(subset=["LawPeriod"])
print(len(seatbelts_monthly), "->", len(clean_seatbelts), "rows after dropping the missing labels")

fig, ax = plt.subplots()
sns.barplot(data=clean_seatbelts, x="month", y="DriversKilled", hue="LawPeriod", errorbar=None, ax=ax)
ax.set(title="Grouped Bar Plot (rows with a missing label removed)", xlabel="Month", ylabel="Mean drivers killed per month")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Scatter plots

We'll get introduced to scatter plots by recreating this graph:

In [ ]:
# Season is a second derived category -- the same confound we uncovered in Session 2's
# regression task (kms vs DriversKilled has a *negative* raw relationship because both
# are driven by season) should be visible here too
month_to_season = {
    "December": "Winter", "January": "Winter", "February": "Winter",
    "March": "Spring", "April": "Spring", "May": "Spring",
    "June": "Summer", "July": "Summer", "August": "Summer",
    "September": "Autumn", "October": "Autumn", "November": "Autumn",
}
seatbelts_monthly["Season"] = pd.Categorical(
    seatbelts_monthly["month"].astype(str).map(month_to_season),
    categories=["Winter", "Spring", "Summer", "Autumn"], ordered=True,
)

fig, ax = plt.subplots()
sns.scatterplot(data=seatbelts_monthly, x="kms", y="DriversKilled", hue="Season", style="Season", ax=ax)
sns.regplot(data=seatbelts_monthly, x="kms", y="DriversKilled", scatter=False, color="black", ax=ax)   # one overall linear fit (+ confidence band)
ax.set(title="Drivers killed and distance driven\nGreat Britain, monthly, 1969-1984",
       xlabel="Distance driven (km)", ylabel="Drivers killed")
plt.show()
# notice the overall trend line slopes DOWN -- more driving, fewer deaths?! Look at how
# the colors cluster: high-kms points skew toward Spring/Summer (fewer deaths), low-kms
# points skew toward Winter (more deaths). The trend line is picking up season, not a
# genuine "more driving is safer" effect. We resolved exactly this in Session 2 by
# controlling for month in a regression.

Building it up step by step *(syntax demo — same plot as above, one layer at a time, using matplotlib directly)*:

In [ ]:
# blank canvas: a figure containing one empty set of axes
fig, ax = plt.subplots()
plt.show()

In [ ]:
# adding the data: x and y, one point per month
fig, ax = plt.subplots()
ax.scatter(seatbelts_monthly["kms"], seatbelts_monthly["DriversKilled"])
plt.show()

In [ ]:
# making it pretty: one colour per season (we draw each season's points separately)
colours = dict(zip(["Winter", "Spring", "Summer", "Autumn"], sns.color_palette("colorblind", 4)))

fig, ax = plt.subplots()
for season, group in seatbelts_monthly.groupby("Season", observed=True):
    ax.scatter(group["kms"], group["DriversKilled"], color=colours[season], label=season)
ax.legend(title="Season")
plt.show()

In [ ]:
# adding the linear model: fit a straight line y = slope * x + intercept to ALL the points and draw it
slope, intercept = np.polyfit(seatbelts_monthly["kms"], seatbelts_monthly["DriversKilled"], deg=1)
x_line = np.array([seatbelts_monthly["kms"].min(), seatbelts_monthly["kms"].max()])

fig, ax = plt.subplots()
for season, group in seatbelts_monthly.groupby("Season", observed=True):
    ax.scatter(group["kms"], group["DriversKilled"], color=colours[season], label=season)
ax.plot(x_line, slope * x_line + intercept, color="black")
ax.legend(title="Season")
plt.show()

In [ ]:
# changing the dots to different shapes, and adding the labels
shapes = {"Winter": "o", "Spring": "s", "Summer": "^", "Autumn": "D"}

fig, ax = plt.subplots()
for season, group in seatbelts_monthly.groupby("Season", observed=True):
    ax.scatter(group["kms"], group["DriversKilled"], color=colours[season], marker=shapes[season], label=season)
ax.plot(x_line, slope * x_line + intercept, color="black")
ax.set_title("Drivers killed and distance driven\nGreat Britain, monthly, 1969-1984")
ax.set_xlabel("Distance driven (km)")
ax.set_ylabel("Drivers killed")
ax.legend(title="Season")
plt.show()

## Regressions & regression tables

We will now use a new data set. `simulated_trust_data` is **entirely simulated** — generated by the code below, not collected from real respondents — built to look like a pilot survey experiment testing the effects of exposure to polarized online discussions (`treatmentgroup`) on **generalized trust** (`gentrust`), alongside two covariates: `pol_id` (a baseline measure of political identification) and `polarisation` (a measure of **affective polarization**). This part of the session isn't about Seatbelts — it's a different kind of dataset (an experiment, not observational time series) that needs its own example.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from IPython.display import HTML, display

# SIMULATED DATA -- for teaching purposes only. Not real experimental data.
#
# What each variable is meant to operationalize:
#  - gentrust: GENERALIZED TRUST. The classic social-capital survey item (GSS/World
#    Values Survey): "Generally speaking, would you say that most people can be
#    trusted, or that you can't be too careful in dealing with people?" -- here on a
#    continuous 0-10 scale (0 = you can't be too careful, 10 = most people can be trusted).
#  - pol_id: political identification / ideological self-placement, a 7-point scale
#    (e.g. 1 = strongly left-leaning, 7 = strongly right-leaning). Simulated here as
#    continuous within [1,7] for simplicity -- a real survey item like this is usually
#    answered as a whole number, not a decimal.
#  - polarisation: AFFECTIVE polarization, NOT perceived/societal polarization. This is
#    about the respondent's own feelings -- warmth toward their political in-group and
#    hostility toward the out-group (the standard construct in political psychology,
#    e.g. Iyengar et al.'s feeling-thermometer-difference measure) -- not a belief about
#    how divided society is in general. Here it's a standardized 0-10 index (higher =
#    more affectively polarized: stronger in-group loyalty / out-group hostility).

def simulate_trust(seed, n=300):
    rng = np.random.default_rng(seed)
    treatmentgroup = rng.binomial(1, 0.5, n)
    pol_id = np.clip(rng.normal(4, 1.5, n), 1, 7)
    polarisation = np.clip(5 + 1.5 * treatmentgroup + 0.1 * pol_id + rng.normal(0, 1.5, n), 0, 10)   # clipped to its 0-10 scale
    gentrust = np.clip(7 - 1.0 * treatmentgroup - 0.3 * polarisation + 0.15 * pol_id + rng.normal(0, 1.2, n), 0, 10)
    return pd.DataFrame({"treatmentgroup": treatmentgroup, "pol_id": pol_id,
                         "polarisation": polarisation, "gentrust": gentrust})

simulated_trust_data = simulate_trust(seed=133)

# let's familiarize with the dataset. What steps would you do?
simulated_trust_data.describe()

In [ ]:
# Models
reg = smf.ols("gentrust ~ treatmentgroup", data=simulated_trust_data).fit()
polid_mod = smf.ols("gentrust ~ treatmentgroup + pol_id", data=simulated_trust_data).fit()
perpol_mod = smf.ols("gentrust ~ treatmentgroup + polarisation", data=simulated_trust_data).fit()
fullmod = smf.ols("gentrust ~ treatmentgroup + polarisation + pol_id", data=simulated_trust_data).fit()
polonly = smf.ols("gentrust ~ polarisation", data=simulated_trust_data).fit()
polidonly = smf.ols("gentrust ~ pol_id", data=simulated_trust_data).fit()

In [ ]:
results = summary_col(
    [reg, polonly, polidonly, polid_mod, perpol_mod, fullmod],
    stars=True,
    float_format="%.3f",
    model_names=["(1)", "(2)", "(3)", "(4)", "(5)", "(6)"],
    regressor_order=["treatmentgroup", "polarisation", "pol_id", "Intercept"],
)
print(results.as_text())

To produce a nicer table in a notebook or a web page, ask for the HTML version:

In [ ]:
display(HTML(results.as_html()))

In theory, this is LaTeX. If you paste this into Overleaf this should turn it into a table.

In [ ]:
print(results.as_latex())

### Task: reading the model comparison

Look at the table above — six columns, six specifications of the same outcome (`gentrust`). Work through this in two stages.

**Stage 1 — read the table on its own terms.**

1. **Track the `treatmentgroup` coefficient** across models (1), (4), (5) and (6) — that is `reg`, `polid_mod`, `perpol_mod` and `fullmod`. Does it stay roughly the same size, or does it change meaningfully when a covariate is added? Which covariate's addition moves it, if any?
   *(A tip, not something to answer yet: keep an eye on the standard error too as you do this. As a rule of thumb, a shrinking SE usually means the added variable explains outcome variance without overlapping much with your treatment — a precision gain. A growing SE can mean the added variable is substantially correlated with the treatment itself. Worth remembering, not something to diagnose here.)*
2. **Track R²** across the same four models. Which one "fits" best by that measure?
3. Based on 1-2 alone — if someone handed you just this table, with no other context — which model would you report as your headline result, and why?

**Stage 2 — the twist.**

4. Here's something the table alone can't tell you: `pol_id` was measured *before* participants were assigned to `treatmentgroup` (a baseline political-identity measure). `polarisation` — affective polarization, the respondent's own in-group warmth / out-group hostility — was plausibly measured *after* exposure to the treatment. If that's right, `polarisation` may be a **post-treatment mediator**: caused by the treatment, and itself a cause of `gentrust`, rather than a safe baseline covariate like `pol_id`.
5. Does that change your answer to (3)? A model can look statistically excellent — high R², a clean sign, a "significant" coefficient — and still be the wrong one to trust, if including a variable risks absorbing part of the very effect you're trying to measure (more formally: violating the ignorability assumption needed to interpret the remaining coefficient causally). Which model would you report now, and why?
6. This should feel familiar: it's the same shape of problem as `kms` and `month` in Session 2's regression task, just with a different mechanism (a mediator here, a confounder there). Both are cases where "which variables to control for" is a design question — something the table's fit statistics can't settle on their own.

### Coefficient plots

Coefficient plots are currently one of the preferred ways to present an experiment's results. Hold on tight, the following code will give us the chance to point out a few more fun things.

In [ ]:
# We refer to the "fullmod" we ran earlier
# Collect its estimates and 95% confidence intervals in a small table
coef_table = pd.DataFrame({
    "estimate": fullmod.params,
    "lower": fullmod.conf_int()[0],      # lower end of the 95% confidence interval
    "upper": fullmod.conf_int()[1],      # upper end
})
print(coef_table.round(3))

# Filter out the intercept
coef_table = coef_table.drop(index="Intercept")

In [ ]:
# Create the coefficient plot with a bold zero line and without intercept
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.errorbar(
    x=coef_table["estimate"], y=coef_table.index,
    xerr=[coef_table["estimate"] - coef_table["lower"], coef_table["upper"] - coef_table["estimate"]],   # distances from the estimate down to / up to the interval ends
    fmt="o", color="black", capsize=4,
)
ax.axvline(0, color="red", linewidth=2)   # the bold zero line
ax.set(title="Coefficient Plot", xlabel="Estimates", ylabel="Predictors")
ax.invert_yaxis()                          # list the first predictor at the top
plt.tight_layout()
plt.show()
# What are the bars? The 95% confidence interval, roughly estimate +/- 1.96 * standard error
# (conf_int() uses the t distribution, so it is a touch wider). NOT +/- 1 SE: that would be only a
# ~68% interval, and then "the bar doesn't cross zero" would no longer mean "significant at 5%".
# Which interval crosses the zero line here -- and what does that tell you about pol_id?

In the next session we are going to learn about alternative modeling strategies. See you then!